<a href="https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
!git clone https://github.com/emgakii001/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 144 (delta 54), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.85 MiB | 11.85 MiB/s, done.
Resolving deltas: 100% (54/54), done.


In [20]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship


In [21]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


In [22]:
print(df["freshness_tier"].value_counts())

freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
Name: count, dtype: int64


In [23]:
print(df["position_tier"].value_counts())
print()
print(df.groupby("position_tier")["ctr"].mean().sort_values())

position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64

position_tier
deep        0.150212
page_3_5    0.222484
striking    0.323239
page_1      0.652467
top_3       1.483611
Name: ctr, dtype: float64


In [24]:
staleness_bucket = df.groupby("freshness_tier").agg(
    n=("content_id", "count"),
    pct_declining=("trend_direction", lambda x: (x == "down").mean() * 100)
).round(1)

print(staleness_bucket)

                    n  pct_declining
freshness_tier                      
0-30            20480           51.1
181+              174           47.1
31-90             175           58.9
91-180           9171           61.1


## **1. My rule and its reason codes**


To build my baseline rule, I tested two signals against the real data. The first signal was **staleness**, based on the belief that pages not updated for a long time are more likely to be declining. The bucket table showed that pages updated within **0–30 days** had a **51.1%** decline rate (**n = 20,480**), **31–90 days** had **58.9%** (**n = 175**), **91–180 days** had the highest decline rate at **61.1%** (**n = 9,171**), while the **181+ day** group dropped to **47.1%** (**n = 174**). The verdict was **MIXED** because decline rates increased with age up to the **91–180 day** bucket but then fell for the oldest pages. This suggests a possible survivorship effect, where very old pages that remain active may be stable evergreen content rather than pages needing refreshes. As a result, I revised my rule to target the **91–180 day** freshness tier instead of **181+**, since it combines the highest observed decline rate with a sufficiently large sample size.

The second signal tested was **CTR versus ranking position**, based on the belief that pages should be compared only against other pages ranking in similar positions. The bucket table showed a clear and consistent increase in average CTR as ranking position improved: **deep (0.15%)**, **page_3_5 (0.22%)**, **striking (0.32%)**, **page_1 (0.65%)**, and **top_3 (1.48%)**. The verdict was **CONFIRMED**, as the relationship followed the expected pattern without reversals, confirming that comparing CTR within the same **position_tier** is a fair and meaningful approach, consistent with Haris's CTR-fix logic.

Based on these findings, my revised rule flags a page for review if it shows evidence of staleness during its highest-risk period or if its click-through rate is unusually low relative to other pages in the same ranking tier. The rule applies only to pages with sufficient search exposure to judge performance reliably, excluding very new or extremely low-impression pages. Pages in the **91–180 day freshness tier** receive the reason code **STALE_CONTENT** because this group recorded the highest decline rate (**61.1%**) with a strong sample size (**n = 9,171**). Pages whose CTR falls meaningfully below the typical CTR for their **position_tier** receive the reason code **LOW_CTR_FOR_POSITION**. Any page meeting either condition is assigned the action label **REVIEW**.


In [25]:

# Signal 1: Staleness bucket table
staleness_bucket = df.groupby("freshness_tier").agg(
    n=("content_id", "count"),
    pct_declining=("trend_direction", lambda x: (x == "down").mean() * 100)
).round(1)

print("SIGNAL 1 — Staleness vs. Decline Rate")
print(staleness_bucket)
print("Verdict: MIXED (peaks at 91-180 days, reverses at 181+ on small n=174)\n")

# Signal 2: CTR by position tier
ctr_bucket = df.groupby("position_tier").agg(
    n=("content_id", "count"),
    mean_ctr=("ctr", "mean")
).round(3).sort_values("mean_ctr")

print("SIGNAL 2 — CTR by Position Tier")
print(ctr_bucket)
print("Verdict: CONFIRMED (clean, monotonic staircase — no reversals)")

SIGNAL 1 — Staleness vs. Decline Rate
                    n  pct_declining
freshness_tier                      
0-30            20480           51.1
181+              174           47.1
31-90             175           58.9
91-180           9171           61.1
Verdict: MIXED (peaks at 91-180 days, reverses at 181+ on small n=174)

SIGNAL 2 — CTR by Position Tier
                   n  mean_ctr
position_tier                 
deep            1319     0.150
page_3_5        7242     0.222
striking        7304     0.323
page_1         11814     0.652
top_3           2321     1.484
Verdict: CONFIRMED (clean, monotonic staircase — no reversals)


In [26]:
# Look at the actual distribution of CTR-vs-tier-average ratio, to pick a defensible cutoff
ratio = df["ctr"] / tier_avg_ctr
print(ratio.describe())
print()
print("Percentile breakdown:")
print(ratio.quantile([0.1, 0.2, 0.3, 0.4, 0.5]))

count    22006.000000
mean         1.000000
std          1.555042
min          0.000000
25%          0.000000
50%          0.561960
75%          1.296653
max         37.534798
Name: ctr, dtype: float64

Percentile breakdown:
0.1    0.000000
0.2    0.000000
0.3    0.140940
0.4    0.366445
0.5    0.561960
Name: ctr, dtype: float64


In [27]:
# Apply the population rule first: exclude pages with too little exposure to judge fairly
eligible = df[df["impressions_90d"] >= 100].copy()

print(f"Pages before population filter: {len(df)}")
print(f"Pages after population filter (impressions_90d >= 100): {len(eligible)}")

# Now check the ratio distribution on ONLY eligible pages
ratio_eligible = eligible["ctr"] / eligible.groupby("position_tier")["ctr"].transform("mean")
print()
print(ratio_eligible.quantile([0.1, 0.2, 0.3, 0.4, 0.5]))

Pages before population filter: 30000
Pages after population filter (impressions_90d >= 100): 22006

0.1    0.000000
0.2    0.000000
0.3    0.140940
0.4    0.366445
0.5    0.561960
Name: ctr, dtype: float64


**2. Build the ranked queue (writes the CSV)**

Encoding the Rule

I initially tested two threshold levels for the low-CTR signal (below 50% and below 30% of the tier average), but both flagged over 70% of all pages — far too loose to be a usable review queue, echoing the "alarm that rings all day" problem from this week's session. Investigating why revealed that many pages had zero clicks simply due to very low impressions, not genuine underperformance — meaning my population filter (excluding low-exposure pages) needed to actually be applied in code, not just stated in the plan. After filtering to pages with at least 100 impressions and checking the real distribution of CTR-vs-tier-average ratios, I set the low-CTR threshold at the 30th percentile (ratio < 0.14) — a cutoff that reflects genuinely underperforming pages rather than noise from too-small a sample.


In [28]:
# Section 2 — full pipeline: population filter, both signals, score, rank, write CSV

# POPULATION: only pages with enough exposure to judge fairly
eligible = df[df["impressions_90d"] >= 100].copy()

# SIGNAL 1: staleness (91-180 day window — the tier with highest, well-supported decline rate)
eligible["flag_stale"] = (eligible["freshness_tier"] == "91-180")

# SIGNAL 2: CTR meaningfully below position-tier peers (bottom ~30th percentile of the ratio)
tier_avg_ctr = eligible.groupby("position_tier")["ctr"].transform("mean")
ctr_ratio = eligible["ctr"] / tier_avg_ctr
eligible["flag_low_ctr"] = ctr_ratio < 0.14

# REASON CODE
def get_reason_code(row):
    if row["flag_stale"] and row["flag_low_ctr"]:
        return "STALE_CONTENT + LOW_CTR_FOR_POSITION"
    elif row["flag_stale"]:
        return "STALE_CONTENT"
    elif row["flag_low_ctr"]:
        return "LOW_CTR_FOR_POSITION"
    else:
        return "NO_FLAG"

eligible["reason_code"] = eligible.apply(get_reason_code, axis=1)
eligible["action"] = eligible["reason_code"].apply(lambda x: "REVIEW" if x != "NO_FLAG" else "MONITOR")

# SCORE: prioritize by size of opportunity (impressions x CTR gap)
eligible["score"] = eligible["impressions_90d"] * (tier_avg_ctr - eligible["ctr"]).clip(lower=0)

# RANK
ranked_queue = eligible.sort_values("score", ascending=False).reset_index(drop=True)

print("Reason code breakdown:")
print(ranked_queue["reason_code"].value_counts())
print()
print("Action breakdown:")
print(ranked_queue["action"].value_counts())
print(f"\n% flagged for review: {(ranked_queue['action']=='REVIEW').mean()*100:.1f}%")

# WRITE THE CSV
import os
os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("\nSaved to work/outputs/baseline_action_score.csv")

Reason code breakdown:
reason_code
NO_FLAG                                 9720
STALE_CONTENT                           5830
LOW_CTR_FOR_POSITION                    4202
STALE_CONTENT + LOW_CTR_FOR_POSITION    2254
Name: count, dtype: int64

Action breakdown:
action
REVIEW     12286
MONITOR     9720
Name: count, dtype: int64

% flagged for review: 55.8%

Saved to work/outputs/baseline_action_score.csv


## **3. Top-20 review**

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Below, I review the twenty highest-scored pages in the ranked queue. Reviewing the top of a ranked list — rather than trusting the rule blindly — is the "human check" step described in this week's session: a rule can compute a score, but only a person can judge whether that score reflects a genuine problem or a false alarm.

Across the twenty pages, three patterns repeat, matching the two signals encoded in the rule: pages flagged purely for staleness (91-180 day freshness tier), pages flagged purely for underperforming CTR relative to their position tier, and pages triggering both signals at once — which represent the highest-confidence cases, since two independent signals agreeing is stronger evidence than either alone.

One pattern worth calling out specifically: row 3 (content_c8e9d6ab9013) records a CTR of exactly 0.00% across 208,678 impressions — not just low, but literally zero. This is unusual enough to suspect a possible technical issue (a broken snippet link, an indexing problem) rather than a pure content-quality problem, and would be worth a quick technical check before assuming a content refresh is the right fix.

For each page below, I note the action, reason code, my confidence in the flag, and — critically — what fact, if true, would make this flag a false alarm.

(row-by-row write-up follows — Rows 0-19, as compiled)

In [29]:
top_20 = ranked_queue.head(20)[["content_id", "score", "reason_code", "action",
                                  "freshness_tier", "position_tier", "ctr",
                                  "impressions_90d"]]
print(top_20.to_string())

              content_id          score                           reason_code   action freshness_tier position_tier   ctr  impressions_90d
0   content_5fe46e04994d  111184.288695                         STALE_CONTENT   REVIEW         91-180        page_1  0.14           517715
1   content_8c19996aa890   93767.338236                               NO_FLAG  MONITOR           0-30         top_3  0.15           509252
2   content_36ff89c8214e   89933.656438                         STALE_CONTENT   REVIEW         91-180        page_1  0.05           295097
3   content_8451fc6f034d   82766.496060                  LOW_CTR_FOR_POSITION   REVIEW           0-30         top_3  0.03           272144
4   content_c8e9d6ab9013   74030.532830  STALE_CONTENT + LOW_CTR_FOR_POSITION   REVIEW         91-180        page_1  0.00           208678
5   content_c84a0ab98e90   72509.410303                  LOW_CTR_FOR_POSITION   REVIEW           0-30        page_1  0.03           223271
6   content_cb112fce36be   

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks**

Although the baseline rule successfully prioritizes many high-opportunity pages, not every page in the ranked queue is equally convincing. Some pages were flagged because they satisfy one rule condition, but the supporting evidence is weaker than for pages that trigger multiple signals.

Row 16 (content_33b4dceecad1) is one of the weakest picks. It was flagged only because it falls within the 91-180 day freshness tier, yet its CTR (0.16%) is noticeably stronger than several other pages in the review queue. Unlike pages that combine staleness with very poor CTR, this page has only one supporting signal, making the recommendation less certain. If the content is evergreen or continues satisfying search intent, refreshing it may provide little benefit.

Row 17 (content_f42eb861c6dd) is another weaker recommendation. Like Row 16, it is driven entirely by the staleness rule rather than multiple independent signals. Although it receives substantial search traffic, there is no evidence from the baseline rule that users are responding poorly to the page. If search demand has simply declined because of seasonality or market trends, refreshing the content may not improve performance.

Row 5 (content_cb112fce36be) is also a moderate-confidence recommendation. It qualifies because it belongs to the highest-risk freshness window identified during the signal audit, but it was not flagged for unusually poor CTR. This means the recommendation depends almost entirely on the assumption that pages between 91 and 180 days are more likely to decline. A manual review should confirm that the content is actually outdated before resources are committed to refreshing it.

Overall, these weaker picks highlight an important limitation of rule-based systems. A single rule can identify potentially valuable pages, but it cannot distinguish between genuinely outdated content and evergreen content that remains accurate. This is one of the reasons a machine learning model is expected to outperform the baseline in later weeks.

In [30]:
# Weak picks — flagged on a single signal only, lower confidence
weak_picks = ranked_queue[ranked_queue["content_id"].isin(
    ["content_33b4dceecad1", "content_f42eb861c6dd", "content_cb112fce36be"]
)][["content_id", "score", "reason_code", "freshness_tier", "position_tier", "ctr", "impressions_90d"]]

print("Weak picks — single-signal flags with lower confidence:")
print(weak_picks.to_string())

Weak picks — single-signal flags with lower confidence:
              content_id         score    reason_code freshness_tier position_tier   ctr  impressions_90d
6   content_cb112fce36be  60357.961033  STALE_CONTENT         91-180        page_1  0.16           309910
23  content_33b4dceecad1  35363.287460  STALE_CONTENT         91-180        page_1  0.16           181574
25  content_f42eb861c6dd  34268.428524  STALE_CONTENT         91-180        page_1  0.13           152467


In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Leakage Check**

The baseline rule was designed to use only information that would have been available at the time the review decision was made. No future-window information or product-generated recommendation fields were used when calculating the review score.

Specifically:

Only the starter dataset's pre-aggregated 90-day trailing window was used to build the baseline rule and rank pages — no warehouse data from later months was involved in this notebook.
The rule uses only observable signals available during that window, including freshness_tier, ctr, position_tier, and impressions_90d.
No product-generated fields such as health_score, priority_score, or action_type were used.
No label-derived variables or future performance measurements were included in the scoring logic.
The review queue therefore represents an honest rule-based baseline that can later be compared fairly against a machine learning model.

This confirms that the baseline is free from both product leakage and future-window leakage, making it a valid benchmark for subsequent modeling work.

In [32]:
# Leakage check — confirm the rule only used legitimate signals

features_used = ["freshness_tier", "ctr", "position_tier", "impressions_90d"]
forbidden_signals = ["trend_direction", "trend_pct", "health_score", "priority_score",
                      "action_type", "is_declining_label"]

print("Features actually used in the rule:")
print(features_used)
print()
print("Checking none of these forbidden signals were used:")
for signal in forbidden_signals:
    used = signal in features_used
    print(f"  {signal}: {'LEAKED — PROBLEM' if used else 'not used — clean'}")

print()
print("Data source: starter dataset (content_refresh_anonymized.csv),")
print("a pre-aggregated 90-day trailing window per page — not warehouse")
print("data from any specific month.")

Features actually used in the rule:
['freshness_tier', 'ctr', 'position_tier', 'impressions_90d']

Checking none of these forbidden signals were used:
  trend_direction: not used — clean
  trend_pct: not used — clean
  health_score: not used — clean
  priority_score: not used — clean
  action_type: not used — clean
  is_declining_label: not used — clean

Data source: starter dataset (content_refresh_anonymized.csv),
a pre-aggregated 90-day trailing window per page — not warehouse
data from any specific month.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.